compare calibration result and simulation result in the instant loop detector

In [41]:
import os
print("Current directory:", os.getcwd())
HORNSGATAN_HOME = os.environ["HORNSGATAN_HOME"]
os.chdir(HORNSGATAN_HOME)
print("Current directory:", os.getcwd())

#number = len(data)
import pandas as pd
date = '2000-01-01'
detector = 'e2w_out'
number = -1
fcd_from_calibration = True
#if fcd_from_calibration:
#    path= "data/calibration_data/"
#else:
#   path = "data/sim_data/"
path = "data/sim_data/"
if number<1:
    postfix = f"{detector}_{date}"
else:
    postfix = f"{detector}_{date}_{number}"

instantInductionLoop_filename_xml = f"{path}instantInductionLoop_{postfix}.xml"


Current directory: /home/kaveh/projects/Hornsgatan
Current directory: /home/kaveh/projects/Hornsgatan


In [24]:
import xml.etree.ElementTree as ET

# Parse the XML content of output_detectors.xml
tree_output = ET.parse(instantInductionLoop_filename_xml)
root_output = tree_output.getroot()

# Extract data from the XML
instant_out_data = []
for instant_out in root_output.findall('instantOut'):
    instant_out_data.append({
        'id': instant_out.get('id'),
        'time': float(instant_out.get('time')),
        'state': instant_out.get('state'),
        'vehID': instant_out.get('vehID'),
        'speed': float(instant_out.get('speed')),  # Convert speed to km/h
        'length': instant_out.get('length'),
        'type': instant_out.get('type'),
        'gap': instant_out.get('gap')
    })

# Convert to a DataFrame
df_instant_out = pd.DataFrame(instant_out_data)
df_instant_out = df_instant_out[df_instant_out["state"]=='enter']
# Save to CSV
#output_csv_file_instant_out = 'output_instant_out.csv'
#df_instant_out.to_csv(output_csv_file_instant_out, index=False)
#print(f"Data from 'output_detectors.xml' successfully converted to CSV and saved as '{output_csv_file_instant_out}'.")


FileNotFoundError: [Errno 2] No such file or directory: 'data/calibration_data/instantInductionLoop_e2w_out_2000-01-01.xml'

In [6]:
df_instant_out

,id,time,state,vehID,speed,length,type,gap
0,e2w_out,1.708316e+09,enter,0_e2w_out,4.63,5.00,DEFAULT_VEHTYPE,None
3,e2w_out,1.708316e+09,enter,1_e2w_out,4.82,5.00,DEFAULT_VEHTYPE,8.03
6,e2w_out,1.708316e+09,enter,2_e2w_out,5.62,5.00,DEFAULT_VEHTYPE,2.55
9,e2w_out,1.708316e+09,enter,3_e2w_out,7.14,5.00,DEFAULT_VEHTYPE,1.62
12,e2w_out,1.708316e+09,enter,4_e2w_out,4.14,5.00,DEFAULT_VEHTYPE,3.30
15,e2w_out,1.708316e+09,enter,5_e2w_out,7.81,5.00,DEFAULT_VEHTYPE,1.54
17,e2w_out,1.708316e+09,enter,6_e2w_out,6.53,5.00,DEFAULT_VEHTYPE,1.52
19,e2w_out,1.708316e+09,enter,7_e2w_out,6.49,5.00,DEFAULT_VEHTYPE,1.62
22,e2w_out,1.708316e+09,enter,8_e2w_out,7.78,5.00,DEFAULT_VEHTYPE,160.66
25,e2w_out,1.708316e+09,enter,9_e2w_out,7.99,5.00,DEFAULT_VEHTYPE,83.44


In [42]:
data = pd.read_csv(f'data/calibration_data/calibrated_data_{postfix}.csv')
data.rename(columns={"veh_id": "vehID"}, inplace=True)
data.head()

,vehID,time_detector_sim,speed_detector_sim,speed_factor,time_detector_real,depart,departSpeed,speed_detector_real,delta_time,delta_speed
0,0_e2w_out,1.708316e+09,5.766496,0.60,1708315885,1708315832,4.9980,5.000000,-0.85,0.766496
1,1_e2w_out,1.708316e+09,4.839190,0.60,1708315894,1708315840,4.9980,1.111111,-1.61,3.728079
2,2_e2w_out,1.708316e+09,4.298588,0.60,1708315895,1708315843,4.9980,0.833333,1.00,3.465254
3,3_e2w_out,1.708316e+09,3.836421,0.60,1708315898,1708315846,4.9980,1.111111,0.96,2.725310
4,4_e2w_out,1.708316e+09,5.988727,1.35,1708315899,1708315865,11.2455,1.666667,1.85,4.322060


In [26]:
print(len(data), len(df_instant_out))

24 24


compare time error and speed error in loop detector between simulation result and calibration result

In [37]:
compare_df = pd.merge(df_instant_out[["vehID","time","speed"]],
            data[["vehID","time_detector_sim","speed_detector_sim"
                  ,"time_detector_real","speed_detector_real"]],on="vehID")
compare_df.head()




,vehID,time,speed,time_detector_sim,speed_detector_sim,time_detector_real,speed_detector_real


In [ ]:
#compare_df = data

In [44]:
time_calib_error = compare_df["time_calib_error"]=(compare_df["time_detector_sim"]-compare_df["time_detector_real"])
abs(time_calib_error).mean()

np.float64(1.2212499876817067)

In [12]:
time_sim_error = compare_df["time_sim_error"] = compare_df["time"]-compare_df["time_detector_real"]
abs(time_sim_error).mean()

np.float64(1.942500005165736)

In [45]:
speed_calib_error= compare_df["speed_calib_error"] =compare_df["speed_detector_sim"]-compare_df["speed_detector_real"]
abs(speed_calib_error).mean()

np.float64(2.337084463461112)

In [14]:
speed_sim_error=compare_df["speed_sim_error"] =compare_df["speed"]-compare_df["speed_detector_real"]
abs(speed_sim_error).mean()

np.float64(2.2646296296296295)

In [15]:
start = min(compare_df['time_sim_error'].min(),compare_df['time_calib_error'].min()) 
end = max(compare_df['time_sim_error'].max(), compare_df['time_calib_error'].max())


In [16]:
import plotly.graph_objects as go

fig = go.Figure()

xbins=dict(
        start=start,  # Bin start
        end=end,    # Bin end
        size=1    # Bin size (width of each bin)
    )
sim_MAE = round(abs(compare_df['time_sim_error']).mean(),2)
calib_MAE = round(abs(compare_df['time_calib_error']).mean(),2)

fig.add_trace(go.Histogram(x=compare_df['time_sim_error'], name=f"time_sim_error --> MAE = {sim_MAE}", opacity=0.75, xbins=xbins))
fig.add_trace(go.Histogram(x=compare_df['time_calib_error'], name=f"time_calib_error  --> MAE = {calib_MAE}", opacity=0.75, xbins=xbins))

fig.update_layout(
    barmode='group',  # Use 'overlay' if you want stacked look
    xaxis_title='time error (s)',
    yaxis_title='Count',
    title=f"error  =  simulate  -  real      ------------         {postfix}"
)

fig.show()
fig.write_html(f"diagram/hist_time_error_{postfix}.html")

In [17]:
print(compare_df['time_sim_error'].min(), compare_df['time_sim_error'].max())
print(compare_df['time_calib_error'].min(), compare_df['time_calib_error'].max())

-3.609999895095825 8.609999895095825
-3.6700000762939453 3.25


In [18]:
print(compare_df['speed_sim_error'].min(), compare_df['speed_sim_error'].max())
print(compare_df['speed_calib_error'].min(), compare_df['speed_calib_error'].max())

-3.3277777777777775 6.1433333333333335
-2.718570776372423 5.0155841817217155


In [19]:
start = min(compare_df['speed_sim_error'].min(),compare_df['speed_calib_error'].min()) 
end = max(compare_df['speed_sim_error'].max(), compare_df['speed_calib_error'].max())

In [20]:
import plotly.graph_objects as go

fig = go.Figure()

xbins=dict(
        start=start,  # Bin start
        end=end,    # Bin end
        size=1    # Bin size (width of each bin)
    )
sim_MAE = round(abs(compare_df['speed_sim_error']).mean(),2)
calib_MAE = round(abs(compare_df['speed_calib_error']).mean(),2)
fig.add_trace(go.Histogram(x=compare_df['speed_sim_error'], name=f"speed_sim_error --> MAE = {sim_MAE}", opacity=0.75, xbins=xbins))
fig.add_trace(go.Histogram(x=compare_df['speed_calib_error'], name=f"speed_calib_error --> MAE = {calib_MAE}", opacity=0.75, xbins=xbins))

fig.update_layout(
    barmode='group',  # Use 'overlay' if you want stacked look
    xaxis_title='Value',
    yaxis_title='Count',
    title=f"error  =  simulate  -  real      ------------         {postfix}"
)

fig.show()
fig.write_html(f"diagram/hist_speed_error_{postfix}.html")

In [21]:
#import os
#os.environ["HORNSGATAN_HOME"]
